# Data Preparation
Tokenize OpenWebText into train / val / test splits.

In [1]:
!pip install -q datasets transformers

In [2]:
import os
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/SigEvict"
DATA_DIR = os.path.join(BASE_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)

TRAIN_TOKENS = 1_000_000_000
VAL_TOKENS = 100_000
TEST_TOKENS = 1_000_000
TOTAL_NEEDED = TRAIN_TOKENS + VAL_TOKENS + TEST_TOKENS

print(f"Target: {TOTAL_NEEDED:,} tokens")

Mounted at /content/drive
Target: 1,001,100,000 tokens


In [3]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
dataset = load_dataset("openwebtext", split="train", streaming=True)

all_tokens = torch.empty(TOTAL_NEEDED, dtype=torch.long)
idx = 0
pbar = tqdm(total=TOTAL_NEEDED, unit="tok")

for example in dataset:
    tokens = tokenizer.encode(example["text"])
    n = min(len(tokens), TOTAL_NEEDED - idx)
    all_tokens[idx:idx+n] = torch.tensor(tokens[:n], dtype=torch.long)
    idx += n
    pbar.update(n)
    if idx >= TOTAL_NEEDED:
        break

pbar.close()
print(f"Collected {idx:,} tokens")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

  0%|          | 0/1001100000 [00:00<?, ?tok/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1217 > 1024). Running this sequence through the model will result in indexing errors


Collected 1,001,100,000 tokens


In [4]:
train_tokens = all_tokens[:TRAIN_TOKENS].clone()
val_tokens = all_tokens[TRAIN_TOKENS:TRAIN_TOKENS + VAL_TOKENS].clone()
test_tokens = all_tokens[TRAIN_TOKENS + VAL_TOKENS:].clone()

del all_tokens

torch.save(train_tokens, os.path.join(DATA_DIR, "owt_train_1B.pt"))
torch.save(val_tokens, os.path.join(DATA_DIR, "owt_val_100K.pt"))
torch.save(test_tokens, os.path.join(DATA_DIR, "owt_test_1M.pt"))

print(f"Train: {len(train_tokens):,}")
print(f"Val: {len(val_tokens):,}")
print(f"Test: {len(test_tokens):,}")

!ls -lh {DATA_DIR}

Train: 1,000,000,000
Val: 100,000
Test: 1,000,000
total 7.5G
-rw------- 1 root root 7.7M Apr 15 00:07 owt_test_1M.pt
-rw------- 1 root root 7.5G Apr 15 00:07 owt_train_1B.pt
-rw------- 1 root root 783K Apr 15 00:07 owt_val_100K.pt
